# Project 3: Toxic Comment Detection on YouTube

## Notebook 1 of 3: Data Extraction

This notebook collects the comments used in the project from the [YouTube Data API v3](https://developers.google.com/youtube/v3/docs/commentThreads/list).

**Contents**
1. Videos selected
2. Functions to extract comments
3. Extraction
4. Summary of the raw data

### 1. Videos selected

Comments are collected from four videos by [Nikocado Avocado](https://www.youtube.com/user/nikocadoavocado), a mukbang creator whose videos attract a large volume of comments, many of them hostile. The four videos cover different types of content (two mukbangs, a break-up announcement and a house tour) and were published between March 2021 and February 2023.

| Video ID | Title | Published |
| --- | --- | --- |
| `d3JkqJycmAg` | We Broke Up | Mar 2021 |
| `d-tOb3SDeRw` | MY NEW HOUSE TOUR – I bought a $2,300,000 Million Dollar Penthouse | Apr 2021 |
| `1GyRS20MUmU` | EXTREME BLUE TAKIS FIRE NOODLES WITH HUNGRY FAT CHICK • Mukbang & Recipe | Dec 2021 |
| `lrVIDuKvyMw` | EXTREME FIRE NOODLE CHALLENGE WITH HUNGRY FAT CHICK • Mukbang & Recipe | Feb 2023 |

In [1]:
# Import libraries
import csv
import glob
import os
import time

import pandas as pd

In [2]:
# Videos to extract comments from
VIDEO_IDS = ['d3JkqJycmAg', 'd-tOb3SDeRw', '1GyRS20MUmU', 'lrVIDuKvyMw']

# Maximum number of comments to extract per video (comments are returned newest first)
MAX_COMMENTS_PER_VIDEO = 10000

# Maximum number of API requests for the whole extraction, to stay within the quota
# Each request costs 1 unit of quota and returns up to 100 comment threads
MAX_REQUESTS = 1000

# Folder where one csv file per video is saved
RAW_DIR = '../data/raw'

# Columns to keep for each comment
FIELDS = ['videoId', 'textDisplay', 'textOriginal', 'parentId', 'authorDisplayName',
          'authorProfileImageUrl', 'authorChannelUrl', 'authorChannelId', 'canRate',
          'viewerRating', 'likeCount', 'publishedAt', 'updatedAt', 'comment_id']

### 2. Functions to extract comments

* The API key is read from the environment variable `YOUTUBE_API_KEY`, so that it is never stored in the notebook.
* Each request returns up to 100 comment threads (the maximum allowed), which keeps the number of requests and the quota used as low as possible. Each request costs 1 unit of the default daily quota of 10,000 units.
* Extraction stops once `MAX_COMMENTS_PER_VIDEO` comments (10,000) have been collected for a video. This keeps the videos balanced, so that no single video dominates the data. Comments are returned newest first, so the most recent comments of each video are collected.
* The number of requests is counted, and extraction stops if the whole run would use more than `MAX_REQUESTS` requests, so that the quota available is never exceeded.
* A short pause is added between requests out of consideration for the server.
* A video that has already been extracted is skipped, so that re-running the notebook does not spend quota again. To refresh the data for a video, delete its csv file and re-run.

In [3]:
'YOUTUBE_API_KEY' in os.environ

True

In [4]:
def get_client():
    """Build the YouTube API client using the key in the environment variable YOUTUBE_API_KEY."""
    from googleapiclient.discovery import build
    return build('youtube', 'v3', developerKey=os.environ['YOUTUBE_API_KEY'])


def parse_thread(thread):
    """Return one row for the top-level comment of a thread and one row for each of its replies."""
    rows = []

    # Top-level comment
    top = thread['snippet']['topLevelComment']
    row = dict(top['snippet'])
    row['parentId'] = None  # top-level comments have no parent
    row['comment_id'] = top['id']
    rows.append(row)

    # Replies (the API returns at most 5 replies per thread here)
    for reply in thread.get('replies', {}).get('comments', []):
        row = dict(reply['snippet'])
        row['comment_id'] = reply['id']
        rows.append(row)

    return rows


def extract_comments(youtube, video_id, csv_file, max_comments, max_requests, pause=0.1):
    """Extract up to max_comments comments of a video, using at most max_requests requests,
    and write them to csv_file. Returns the number of comments and the number of requests used."""
    n_comments = 0
    n_requests = 0
    page_token = None
    tmp_file = csv_file + '.part'  # write to a temporary file so a failed run is not mistaken for a complete one

    with open(tmp_file, 'w', encoding='utf8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS, extrasaction='ignore')
        writer.writeheader()
        while n_requests < max_requests:
            response = youtube.commentThreads().list(
                part='snippet,replies',
                videoId=video_id,
                maxResults=100,
                order='time',
                textFormat='plainText',
                pageToken=page_token
            ).execute()
            n_requests += 1

            for thread in response['items']:
                rows = parse_thread(thread)[:max_comments - n_comments]  # do not go over the limit
                writer.writerows(rows)
                n_comments += len(rows)
                if n_comments >= max_comments:
                    break

            page_token = response.get('nextPageToken')
            if not page_token or n_comments >= max_comments:
                break
            time.sleep(pause)

    os.replace(tmp_file, csv_file)
    return n_comments, n_requests


def get_comments(video_ids, max_comments=MAX_COMMENTS_PER_VIDEO, max_requests=MAX_REQUESTS):
    """Extract comments for each video that does not have a csv file yet, within the request budget."""
    os.makedirs(RAW_DIR, exist_ok=True)
    youtube = None
    requests_used = 0
    for video_id in video_ids:
        csv_file = f'{RAW_DIR}/comments_{video_id}.csv'
        if os.path.isfile(csv_file):
            print(f'{video_id}: already extracted, skipped')
            continue
        if requests_used >= max_requests:
            print(f'{video_id}: not extracted, the limit of {max_requests} requests has been reached')
            continue
        if youtube is None:
            youtube = get_client()
        n_comments, n_requests = extract_comments(youtube, video_id, csv_file, max_comments, max_requests - requests_used)
        requests_used += n_requests
        print(f'{video_id}: {n_comments} comments extracted using {n_requests} requests')
    print(f'Total requests used: {requests_used}')

### 3. Extraction

In [5]:
get_comments(VIDEO_IDS)

d3JkqJycmAg: 10000 comments extracted using 63 requests
d-tOb3SDeRw: 10000 comments extracted using 60 requests
1GyRS20MUmU: 10000 comments extracted using 74 requests
lrVIDuKvyMw: 10000 comments extracted using 72 requests
Total requests used: 269


**Comments**

* The four files in `data/raw` were extracted on 14 October 2026, using 269 of the 1,000 requests allowed (60 to 74 requests per video).
* Each video reached the limit of 10,000 comments.
* The warning above is a notice from Google's library that it will stop supporting Python 3.10 in future. It does not affect the extraction.

### 4. Summary of the raw data

In [6]:
# Number of comments, share of replies and date range of each raw file
summary = []
for csv_file in sorted(glob.glob(f'{RAW_DIR}/comments_*.csv')):
    raw = pd.read_csv(csv_file, low_memory=False)
    summary.append({
        'video_id': raw['videoId'].iloc[0],
        'comments': len(raw),
        'replies': raw['comment_id'].str.contains('.', regex=False).sum(),  # reply ids have the form <parent id>.<reply id>
        'columns': raw.shape[1],
        'file_size_mb': round(os.path.getsize(csv_file) / 1e6, 1)
    })

summary = pd.DataFrame(summary)
summary

,video_id,comments,replies,columns,file_size_mb
0,1GyRS20MUmU,10000,2688,14,4.1
1,d-tOb3SDeRw,10000,4073,14,4.4
2,d3JkqJycmAg,10000,3794,14,4.4
3,lrVIDuKvyMw,10000,2843,14,4.4


In [7]:
print(f"Total number of comments: {summary['comments'].sum()}")

Total number of comments: 40000


**Comments**

* A total of 40,000 comments with 14 columns were collected, 10,000 from each of the four videos, so no single video dominates the data.
* 13,398 of the comments (33%) are replies, ranging from 27% of the comments for `1GyRS20MUmU` to 41% for `d-tOb3SDeRw`.
* This is a sufficient amount of text data for training a classification model. The comments are cleaned, explored and labeled in Notebook 2.